# Lab 3: Probability, Distribution Fitting & Simulation

**Course:** INS-605: Data Analysis II  
**Lecturer:** Sothea HAS, PhD

---

**Student name:** ...  
**ID:** ...

### Objective

In this lab, we use real NYC 311 complaint data to connect a simple data-analysis workflow with probability:

1. Select the relevant observations and organize them into time intervals.
2. Visualize a random variable.
3. Pick a reasonable distribution family.
4. Estimate its parameter(s) and compare the observed data with the probability model.
5. Compute probabilities from the fitted model.
6. Simulate website traffic using a transition matrix and decide where an advertisement could be placed.

> **Time:** about 1 hour in class. The homework section extends the analysis.

## 0. Load and understand the NYC 311 data

The data come from **NYC Open Data – 311 Service Requests**. The downloaded period covers **30 August through 1 September 2026** (the end date is exclusive).

For this lab, we are **not** interested in all 311 complaints. We will keep only complaints handled by the **New York City Police Department** and related to **noise**.

### Columns to look at

Pay particular attention to:

- `created_date` — when the complaint was created. **Use this column to organize complaints into time intervals.**
- `agency_name` — the full name of the responsible agency. We want **`New York City Police Department`**.
- `complaint_type` — the main complaint category. We want complaint types beginning with **`Noise`**.
- `descriptor` — a more detailed description of the complaint, such as `Loud Music/Party`.
- `location_type` — where the complaint was reported.

We will mainly use **`created_date`**, **`agency_name`**, and **`complaint_type`** in this lab.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from urllib.parse import quote

target_date = "2026-08-30"
end_date = "2026-09-01"

dataset_id = "erm2-nwe9"

where_clause = (
    f"created_date >= '{target_date}T00:00:00' "
    f"AND created_date < '{end_date}T00:00:00'"
)

url = (
    f"https://data.cityofnewyork.us/resource/{dataset_id}.csv"
    f"?$where={quote(where_clause)}&$limit=50000"
)

data = pd.read_csv(url)

print(f"Downloaded {len(data):,} records.")
print(data.shape)
data.head()

Downloaded 10,535 records.
(10535, 44)


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,70248760,2026-08-31T01:59:39.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11412.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.694215,-73.753087,POINT (-73.753087331085 40.694214514285)
1,70243572,2026-08-31T01:50:52.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,10472.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.827242,-73.874057,POINT (-73.874056693475 40.827242433541)
2,70239573,2026-08-31T01:50:48.000,NaN,DOHMH,Department of Health and Mental Hygiene,Smoking or Vaping,Allowed in Smoke Free Area,Cannabis Smoking or Vaping,Residential Building,11221.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.693773,-73.915534,POINT (-73.91553395065 40.693773172836)
3,70249101,2026-08-31T01:50:26.000,NaN,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,NaN,Club/Bar/Restaurant,11226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.654245,-73.952867,POINT (-73.952867166126 40.654245046963)
4,70248458,2026-08-31T01:50:21.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,11354.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.766446,-73.829089,POINT (-73.829089290124 40.766446350276)


### Question 0a — Inspect the important columns

Display the following columns:

`created_date`, `agency_name`, `complaint_type`, `descriptor`, `location_type`

Then check how many different values appear in `agency_name` and `complaint_type`.

**Question:** Which columns will you need to answer the questions in this lab?

In [2]:
# TODO
# Display the important columns and inspect the unique values.

In [3]:
print(data[["created_date", "agency_name", "complaint_type", "descriptor", "location_type"]].head())
print("Number of agencies:", data["agency_name"].nunique())
print("Number of complaint types:", data["complaint_type"].nunique())

              created_date                              agency_name  \
0  2026-08-31T01:59:39.000             Department of Transportation   
1  2026-08-31T01:50:52.000          New York City Police Department   
2  2026-08-31T01:50:48.000  Department of Health and Mental Hygiene   
3  2026-08-31T01:50:26.000          New York City Police Department   
4  2026-08-31T01:50:21.000          New York City Police Department   

            complaint_type                  descriptor         location_type  
0         Street Condition                     Pothole                   NaN  
1  Noise - Street/Sidewalk            Loud Music/Party       Street/Sidewalk  
2        Smoking or Vaping  Allowed in Smoke Free Area  Residential Building  
3       Noise - Commercial            Loud Music/Party   Club/Bar/Restaurant  
4  Noise - Street/Sidewalk            Loud Music/Party       Street/Sidewalk  
Number of agencies: 13
Number of complaint types: 128


In [4]:
# TODO

# noise = data[
#     (data["agency_name"] == "New York City Police Department")
#     & (data["complaint_type"].str.startswith("Noise", na=False))
# ].copy()

# print(noise.shape)
# noise.head()

In [5]:
noise = data[
    (data["agency_name"] == "New York City Police Department")
    & (data["complaint_type"].str.startswith("Noise", na=False))
].copy()

print("Filtered shape:", noise.shape)
print(noise[["created_date", "agency_name", "complaint_type", "descriptor"]].head())

Filtered shape: (3873, 44)
              created_date                      agency_name  \
1  2026-08-31T01:50:52.000  New York City Police Department   
3  2026-08-31T01:50:26.000  New York City Police Department   
4  2026-08-31T01:50:21.000  New York City Police Department   
6  2026-08-31T01:49:22.000  New York City Police Department   
8  2026-08-31T01:48:18.000  New York City Police Department   

            complaint_type        descriptor  
1  Noise - Street/Sidewalk  Loud Music/Party  
3       Noise - Commercial  Loud Music/Party  
4  Noise - Street/Sidewalk  Loud Music/Party  
6       Noise - Commercial  Loud Music/Party  
8      Noise - Residential      Loud Talking  


In [6]:
# TODO
# Convert created_date to datetime.
# Then create a 30-minute time interval using floor("30min").
#
# noise["created_date"] = pd.to_datetime(noise["created_date"])
# noise["interval"] = noise["created_date"].dt.floor("30min")
#
# Count complaints in each interval.
# interval_counts = noise.groupby("interval").size()

In [7]:
noise["created_date"] = pd.to_datetime(noise["created_date"])
noise["interval"] = noise["created_date"].dt.floor("30min")
interval_counts = noise.groupby("interval").size()

In [8]:
# TODO

In [9]:
print("Number of observed intervals:", len(interval_counts))
print("Minimum complaints:", interval_counts.min())
print("Maximum complaints:", interval_counts.max())
interval_counts.head(10)

Number of observed intervals: 52
Minimum complaints: 9
Maximum complaints: 320


interval
2026-08-30 00:00:00    320
2026-08-30 00:30:00    256
2026-08-30 01:00:00    220
2026-08-30 01:30:00    164
2026-08-30 02:00:00    127
2026-08-30 02:30:00    122
2026-08-30 03:00:00     87
2026-08-30 03:30:00     83
2026-08-30 04:00:00    109
2026-08-30 04:30:00     61
dtype: int64

In [10]:
# TODO

In [11]:
full_index = pd.date_range(
    start=interval_counts.index.min(),
    end=interval_counts.index.max(),
    freq="30min"
)
counts = interval_counts.reindex(full_index, fill_value=0)
counts.name = "complaints"

print("Number of complete intervals:", len(counts))
counts.head()

Number of complete intervals: 52


2026-08-30 00:00:00    320
2026-08-30 00:30:00    256
2026-08-30 01:00:00    220
2026-08-30 01:30:00    164
2026-08-30 02:00:00    127
Freq: 30min, Name: complaints, dtype: int64

In [12]:
# TODO
# Hint: px.histogram(...)

In [13]:
fig = px.histogram(
    counts,
    nbins=20,
    labels={"value": "Complaints per 30-minute interval"},
    title="NYPD Noise Complaints per 30-Minute Interval"
)
fig.update_yaxes(title="Number of intervals")
fig.show()

In [14]:
# No code is required here.
# Write your answer in the Markdown cell above.

## 4. Estimate the parameter and generate the model PMF

For a Poisson distribution, the parameter is `λ`.

A simple estimate is the sample mean:

$$
\hat{\lambda} = \bar{x}
$$

### Question 4a

Calculate `lambda_hat` from `counts`.

Then generate the Poisson probability mass function (PMF) for the range of observed counts.

**Hint:** `stats.poisson.pmf(k, mu=lambda_hat)`

In [15]:
lambda_hat = counts.mean()
k = np.arange(counts.min(), counts.max() + 1)
pmf = stats.poisson.pmf(k, mu=lambda_hat)

print("Estimated lambda:", round(lambda_hat, 3))

Estimated lambda: 74.481


### Question 4b — Compare observed frequencies with the fitted model

Calculate the **observed proportion** for each possible count and compare it with the fitted Poisson PMF.

Create a bar chart for the observed proportions and add the fitted Poisson PMF as points/lines.

**Important:** For a discrete variable, we use a **PMF**, not a continuous PDF.

In [16]:
observed_prob = counts.value_counts(normalize=True).reindex(k, fill_value=0)

fig = go.Figure()
fig.add_trace(go.Bar(x=k, y=observed_prob.values, name="Observed proportion"))
fig.add_trace(go.Scatter(
    x=k, y=pmf, mode="lines+markers", name="Fitted Poisson PMF"
))
fig.update_layout(
    title="Observed Complaint Counts vs Fitted Poisson Model",
    xaxis_title="Complaints per 30-minute interval",
    yaxis_title="Probability"
)
fig.show()

## 5. Compute probabilities

Now use the fitted Poisson model to answer questions about future 30-minute intervals.

### Question 5a

Using your estimated `lambda_hat`, calculate:

$$
P(X = 10)
$$

That is:

> What is the model probability that a randomly selected 30-minute interval contains exactly 10 NYPD noise complaints?

In [17]:
p_exact_10 = stats.poisson.pmf(10, mu=lambda_hat)
print("P(X = 10) =", round(p_exact_10, 4))

P(X = 10) = 0.0


### Question 5b

Calculate:

$$
P(X \geq 10)
$$

**Hint:** For a Poisson model,

`P(X >= 10) = 1 - P(X <= 9)`

Use the CDF (`stats.poisson.cdf`).

Then explain in one sentence what this probability means in the context of the NYC noise complaints.

In [18]:
p_ge_10 = 1 - stats.poisson.cdf(9, mu=lambda_hat)
print("P(X >= 10) =", round(p_ge_10, 4))

P(X >= 10) = 1.0


# 6. Network traffic and ad placement

Now we move from a count distribution to **simulation**.

Imagine a website with four states:

- **Home**
- **Product**
- **Cart**
- **Exit**

A visitor moves from one state to another according to the transition matrix below.

Each row gives the probability of the **next state**, given the current state.

| From / To | Home | Product | Cart | Exit |
|---|---:|---:|---:|---:|
| Home | 0.10 | 0.75 | 0.05 | 0.10 |
| Product | 0.20 | 0.45 | 0.25 | 0.10 |
| Cart | 0.05 | 0.20 | 0.25 | 0.50 |
| Exit | 0.00 | 0.00 | 0.00 | 1.00 |

This is a simple **transition-matrix / Markov-chain** simulation.

### Question 6a

Create the transition matrix in Python and check that every row sums to 1.

In [19]:
states = ["Home", "Product", "Cart", "Exit"]

P = np.array([
    [0.10, 0.75, 0.05, 0.10],
    [0.20, 0.45, 0.25, 0.10],
    [0.05, 0.20, 0.25, 0.50],
    [0.00, 0.00, 0.00, 1.00]
])

transition = pd.DataFrame(P, index=states, columns=states)
transition

,Home,Product,Cart,Exit
Home,0.10,0.75,0.05,0.1
Product,0.20,0.45,0.25,0.1
Cart,0.05,0.20,0.25,0.5
Exit,0.00,0.00,0.00,1.0


### Question 6b — Simulate one visitor

Start at **Home** and repeatedly choose the next state using the transition probabilities.

Stop when the visitor reaches **Exit** or after 20 steps.

**Hint:** `np.random.choice(states, p=...)`.

Write a function `simulate_visit()` that returns the sequence of states.

In [20]:
# TODO

def simulate_visit(max_steps=20):
    # Start at Home
    # At each step, use the appropriate row of P
    # Stop at Exit or max_steps
    pass

simulate_visit()

In [21]:
def simulate_visit(max_steps=20):
    current = "Home"
    visit = [current]

    for _ in range(max_steps - 1):
        if current == "Exit":
            break
        current_index = states.index(current)
        current = np.random.choice(states, p=P[current_index])
        visit.append(current)
        if current == "Exit":
            break

    return visit

simulate_visit()

['Home', np.str_('Exit')]

In [22]:
n_visitors = 5000
all_visits = []

for _ in range(n_visitors):
    all_visits.extend(simulate_visit())

visit_counts = pd.Series(all_visits).value_counts().reindex(states, fill_value=0)

print(visit_counts)

# TODO: create a bar chart

Home        8959
Product    14210
Cart        5292
Exit        4930
Name: count, dtype: int64


In [23]:
fig = px.bar(
    x=visit_counts.index,
    y=visit_counts.values,
    labels={"x": "Website state", "y": "Number of visits"},
    title="Simulated Website Traffic"
)
fig.show()

# Homework — Extend the analysis

Complete the following outside class.

### Homework 1 — Try a different time interval

Repeat the complaint analysis using **15-minute intervals** instead of 30-minute intervals.

1. Create the new count variable.
2. Visualize it.
3. Estimate the Poisson parameter.
4. Compute `P(X >= 5)`.
5. Compare the result with your 30-minute analysis.

**Question:** How does changing the interval length affect the estimated number of complaints per interval?

### Homework 2 — Inter-arrival times

Return to the filtered NYPD noise complaints and use `created_date` directly.

1. Sort complaints chronologically.
2. Calculate the time between consecutive complaints in minutes.
3. Visualize the inter-arrival times.
4. Try an **Exponential** model.
5. Estimate its rate parameter using the sample mean.
6. Compare the fitted Exponential model with the observed data.
7. Compute `P(T > 10)`.

### Homework 3 — Better ad placement

Modify the website simulation so that you record **transitions**, such as `Home → Product`.

1. Count the most common transitions.
2. Suppose an ad can be shown **when entering a state**.
3. Choose one transition/state where you would place the ad.
4. Explain your choice in 2–3 sentences.

> **Submit:** your completed notebook with code, plots, and short written answers.